In [ ]:
import torch
import numpy as np
import numpy.typing as npt
import pandas as pd
from pathlib import Path
import re
import matplotlib.pyplot as plt

In [ ]:
# input should be attention_weights from AggregateThenClassify forward output
def attcord_patch_map(
    att_tensor: torch.Tensor, # att_tensor[i] = attn_weight for token[i]
    coords: torch.Tensor,
    img_size: tuple[int, int, int]
) -> npt.NDArray[np.float32]:
    image = np.zeros(img_size, dtype = np.float32)
    

    coords = np.asarray(coords, dtype=np.int64)
    att_tensor = att_tensor.detach().cpu().numpy().astype(np.float32)
    image[coords[:,0], coords[:,1], coords[:,2]] = att_tensor
    
    return image

def att_pixel_map( #call after calling attcord_patch_map
    patch_map: np.ndarray,
    patch_size: tuple[int, int, int] = (4, 16, 16) #size found from preprocess.py
) -> np.ndarray:
    return patch_map.repeat(patch_size[0], axis=0).repeat(patch_size[1], axis=1).repeat(patch_size[2],axis=2)

# output of att_pixel map should look like attention per pixel
##########################################################  
# attn_weight(0,0) attn_weight(0,1) ... attn_weight(0,n) #
# attn_weight(1,0) attn_weight(1,1) ... attn_weight(1,n) #
# .                                                      #
# .                                                      #
# .                                                      #
# attn_weight(n,0) attn_weight(n,1) ... attn_weight(n,n) #

In [20]:
coords_path = "/Users/deborahfu/eecs281/eecs545/Project/train_coords/batch_1"
coords_path = Path(coords_path)
attn_path = "/Users/deborahfu/eecs281/eecs545/Project/attention_maps"
attn_path = Path(attn_path)

coords_list = []
attn_list = []

coords_pattern = re.compile(r"sub-(\d+)_encoder_coords\.pt")
attn_pattern = re.compile(r"subject_(.+)_attention\.csv")

for file in coords_path.glob("*.pt"):
        match = coords_pattern.search(file.name)
        if match:
            subj_id = match.group(1)
            coords_tensor = torch.load(file, map_location="cpu")
            coords_list.append(
                {
                    "subject id": subj_id,       
                    "coords": coords_tensor,
                }
            )

coords_df = pd.DataFrame(coords_list)


for file in attn_path.glob("*.csv"):
        match = attn_pattern.search(file.name)
        if match:
            subj_id = match.group(1)
            attn_tensor = pd.read_csv(file)
            attn_tensor = torch.tensor(attn_tensor["attention_weight"].values, dtype = torch.float32)
            
            attn_list.append(
                {
                    "subject id": subj_id,       
                    "attns": attn_tensor,
                }
            )

attn_df = pd.DataFrame(attn_list)



In [21]:
def normalize_subject_id(s: str) -> str:
    s = str(s)

    # remove file-derived wrappers if present
    s = re.sub(r"_encoder_coords$", "", s)
    s = re.sub(r"_encoder_embeddings$", "", s)
    s = re.sub(r"_attention$", "", s)

    # remove leading subject_ or sub-
    s = re.sub(r"^subject_", "", s)
    s = re.sub(r"^sub-", "", s)

    return s

coords_df["merge_id"] = coords_df["subject id"].map(normalize_subject_id)
attn_df["merge_id"] = attn_df["subject id"].map(normalize_subject_id)

merged_df = coords_df.merge(
    attn_df,
    on="merge_id",
    how="inner",
    suffixes=("_coords", "_attn")
)

merged_df = merged_df.rename(columns={"merge_id": "subject_id"})
merged_df = merged_df.drop(columns=["subject id_coords", "subject id_attn"])

In [22]:
print(merged_df)

                                              coords    subject_id  \
0  [[tensor(0), tensor(0), tensor(0)], [tensor(0)...  119139956064   
1  [[tensor(0), tensor(0), tensor(0)], [tensor(0)...  117730404599   
2  [[tensor(0), tensor(0), tensor(0)], [tensor(0)...  127728266565   

                                               attns  
0  [tensor(5.0160e-07), tensor(9.7752e-08), tenso...  
1  [tensor(2.5321e-06), tensor(7.8273e-07), tenso...  
2  [tensor(2.9286e-07), tensor(6.3792e-08), tenso...  


In [ ]:
img_size = (100, 100, 100) # substitute in actual image voxel size
merged_df["attn_by_pixel"] = merged_df.apply(
    lambda row: att_pixel_map(
        attcord_patch_map(
            row["attns"],
            row["coords"],
            img_size
        )
    ),
    axis = 1
)


In [24]:
print(merged_df)

                                              coords    subject_id  \
0  [[tensor(0), tensor(0), tensor(0)], [tensor(0)...  119139956064   
1  [[tensor(0), tensor(0), tensor(0)], [tensor(0)...  117730404599   
2  [[tensor(0), tensor(0), tensor(0)], [tensor(0)...  127728266565   

                                               attns  \
0  [tensor(5.0160e-07), tensor(9.7752e-08), tenso...   
1  [tensor(2.5321e-06), tensor(7.8273e-07), tenso...   
2  [tensor(2.9286e-07), tensor(6.3792e-08), tenso...   

                                       attn_by_pixel  
0  [[[5.015967e-07, 5.015967e-07, 5.015967e-07, 5...  
1  [[[2.5320817e-06, 2.5320817e-06, 2.5320817e-06...  
2  [[[2.9285525e-07, 2.9285525e-07, 2.9285525e-07...  


In [ ]:
def normalize_map(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    xmin = np.nanmin(x)
    xmax = np.nanmax(x)
    if xmax > xmin:
        return (x - xmin) / (xmax - xmin)
    return np.zeros_like(x, dtype=np.float32)

In [ ]:
# Working on overlaying attention on the slices


def overlay_attention_on_slice(
    mri_volume: np.ndarray,
    attn_voxel_map: np.ndarray,
    slice_idx: int,
    axis: int = 0,
    alpha: float = 0.4,
    threshold: float | None = None,
    cmap: str = "hot"
):
    if axis == 0:
        mri_slice = mri_volume[slice_idx, :, :]
        attn_slice = attn_voxel_map[slice_idx, :, :]
    elif axis == 1:
        mri_slice = mri_volume[:, slice_idx, :]
        attn_slice = attn_voxel_map[:, slice_idx, :]
    elif axis == 2:
        mri_slice = mri_volume[:, :, slice_idx]
        attn_slice = attn_voxel_map[:, :, slice_idx]
    else:
        raise ValueError("axis must be 0, 1, or 2")

    attn_slice = normalize_map(attn_slice)

    if threshold is not None:
        attn_slice = np.where(attn_slice >= threshold, attn_slice, np.nan)

    plt.figure(figsize=(6, 6))
    plt.imshow(mri_slice, cmap="gray")
    plt.imshow(attn_slice, cmap=cmap, alpha=alpha)
    plt.axis("off")
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.show()